# Step 6 — Semantic Map Encoder (LSTM)

This notebook extends the BiGRU + CGConv trajectory model with a semantic lane encoder based on the Argoverse map API.

Pipeline:

`past_traj → BiGRU encoder → CGConv interaction → target_feature (128)`

`semantic_lane → LSTM semantic encoder → semantic_feature (64)`

`[target_feature ; semantic_feature] → decoder → future trajectory (30,2)`

In [ ]:
# 1. Clone project (Colab) and set PROJECT_ROOT (same pattern as other notebooks)
import os
import sys

REPO_URL = "https://github.com/PulockDas/Implement-STAST-System.git"
BRANCH = "master"
CLONE_DIR = "/content/Implement-STAST-System"  # Colab; use os.getcwd() for local

ip = get_ipython()
if os.path.isdir(CLONE_DIR) and os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    ip.run_line_magic("cd", CLONE_DIR)
    ip.system("git fetch origin")
    ip.system("git clean -fd")
    ip.system(f"git checkout {BRANCH}")
    ip.system(f"git pull origin {BRANCH}")
    print(f"Updated existing repo at {CLONE_DIR} (branch {BRANCH})")
else:
    parent = os.path.dirname(CLONE_DIR) or "."
    os.makedirs(parent, exist_ok=True)
    ip.run_line_magic("cd", parent)
    ip.system(f"git clone --branch {BRANCH} {REPO_URL} {os.path.basename(CLONE_DIR)}")
    print(f"Cloned repo to {CLONE_DIR} (branch {BRANCH})")

PROJECT_ROOT = CLONE_DIR
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
ip.run_line_magic("cd", PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
# 1b. Local fallback: set PROJECT_ROOT if not running in Colab clone cell
import os
import sys

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

In [ ]:
# 2. Mount Google Drive (for loading/saving checkpoints)
from google.colab import drive

drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/stast/checkpoints"
print("Checkpoint dir:", CHECKPOINT_DIR)

In [ ]:
# 3. Argoverse map API (clone repo, use map_files from Drive)
!git clone https://github.com/argoai/argoverse-api.git
import sys
sys.path.append("/content/argoverse-api")

from argoverse.map_representation.map_api import ArgoverseMap

map_root = "/content/drive/MyDrive/stast/dataset/map_files"
avm = ArgoverseMap(root=map_root)

print("Cities:", avm.city_name_to_city_id_dict)

In [ ]:
# 4. Dataset and DataLoader (reuse scene-level dataset)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from pathlib import Path

from datasets.argoverse_dataset import ArgoverseSceneDataset
from utils.config import PAST_STEPS, FUTURE_STEPS

# Assume dataset already downloaded (see previous training notebook) or download again via kagglehub if needed.
import kagglehub

path = kagglehub.dataset_download("narendarmallireddy/argoverse1-motion-dataset")
print("Dataset path:", path)

data_path = Path(path)
csv_files = list(data_path.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV file(s).")
if not csv_files:
    raise FileNotFoundError("No CSV files found under dataset path.")

max_scenes = 5000  # adjust if Colab is slow
scene_csvs = csv_files[:max_scenes]

full_dataset = ArgoverseSceneDataset(
    scene_csvs,
    past_steps=PAST_STEPS,
    future_steps=FUTURE_STEPS,
    max_vehicles=20,
    distance_threshold=50.0,
)

val_fraction = 0.1
val_size = max(1, int(len(full_dataset) * val_fraction))
train_size = max(1, len(full_dataset) - val_size)
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Total scenes:", len(full_dataset))
print("Train scenes:", len(train_dataset))
print("Val scenes:", len(val_dataset))
print("Device:", device)

In [ ]:
# 5. Inspect one batch and map API usage (avm from cell above)
from visualization import plot_trajectory

batch = next(iter(train_loader))
print("batch keys:", batch.keys())
print("past_traj shape:", batch["past_traj"].shape)
print("future_target shape:", batch["future_target"].shape)
print("vehicle_mask shape:", batch["vehicle_mask"].shape)

# Example: query lane IDs near target vehicle last position for first sample
past = batch["past_traj"]  # (B, N, 20, 2)
B, N, T, _ = past.shape

b = 0
x, y = past[b, 0, -1].tolist()  # target vehicle last observed position (currently normalized coords)

# For map queries we need map coordinates and city name; here we show API usage with a fixed city
city_name = "PIT"  # replace with real city_name when available
lane_ids = avm.get_lane_ids_in_xy_bbox(x, y, city_name)
print("lane_ids (example):", lane_ids)

if lane_ids:
    lane_id = lane_ids[0]
    centerline = avm.get_lane_segment_centerline(lane_id, city_name)
    print("example centerline points (first 5):", centerline[:5])

In [ ]:
# 6. Helper to build semantic lane tensors for a batch (avm from cell 3)
import numpy as np

MAX_LANE_POINTS = 20


def build_semantic_lane_batch(batch, avm, city_name: str = "PIT"):
    """Build semantic_lane tensor (B, L, 2) from ArgoverseMap.

    This example uses a fixed city_name; in a full implementation, you
    would use the true city_name from each scenario.
    """
    past = batch["past_traj"]  # (B, N, 20, 2)
    B, N, T, _ = past.shape

    lanes = np.zeros((B, MAX_LANE_POINTS, 2), dtype=np.float32)

    for b in range(B):
        x, y = past[b, 0, -1].tolist()
        lane_ids = avm.get_lane_ids_in_xy_bbox(x, y, city_name)
        if not lane_ids:
            continue
        lane_id = lane_ids[0]
        centerline = avm.get_lane_segment_centerline(lane_id, city_name)
        pts = np.array([[p[0], p[1]] for p in centerline], dtype=np.float32)
        if len(pts) == 0:
            continue
        # Normalize relative to vehicle position
        pts = pts - np.array([x, y], dtype=np.float32)
        # Pad / truncate to MAX_LANE_POINTS
        if len(pts) >= MAX_LANE_POINTS:
            pts_fixed = pts[:MAX_LANE_POINTS]
        else:
            pts_fixed = np.zeros((MAX_LANE_POINTS, 2), dtype=np.float32)
            pts_fixed[: len(pts)] = pts
        lanes[b] = pts_fixed

    return torch.from_numpy(lanes)  # (B, L, 2)

In [ ]:
# 7. Semantic encoder + graph model with fusion
from models.temporal_encoder import BiGRUGraphTrajectoryEncoder
from models.semantic_encoder import SemanticLaneEncoder, GraphSemanticTrajectoryModel

# Load pretrained BiGRU+Graph checkpoint
base_graph_model = BiGRUGraphTrajectoryEncoder().to(device)
ckpt_path = os.path.join(CHECKPOINT_DIR, "bigru_graph_model.pth")
print("Loading base graph checkpoint from:", ckpt_path)
state = torch.load(ckpt_path, map_location=device)
base_graph_model.load_state_dict(state)

# Build fused model
torch.cuda.empty_cache()
model = GraphSemanticTrajectoryModel(base_graph_model).to(device)

# Quick shape test
batch = next(iter(train_loader))
semantic_lane = build_semantic_lane_batch(batch, avm)  # (B, L, 2)

past_scene = batch["past_traj"].to(device)
vehicle_mask = batch["vehicle_mask"].to(device)
semantic_lane = semantic_lane.to(device)

with torch.no_grad():
    pred = model(past_scene, vehicle_mask, semantic_lane)
print("pred shape (with semantic):", tuple(pred.shape), "expected: (B, 30, 2)")

In [ ]:
# 8. Training loop for BiGRU+Graph+Semantic model
from metrics.trajectory_metrics import ade, fde

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

metrics_log_semantic = []
EPOCHS = 5  # 5–10 epochs as a starting point

for epoch in range(EPOCHS):
    # -------- train --------
    model.train()
    total_loss = 0.0
    total_ade = 0.0
    total_fde = 0.0
    n_batches = 0

    for batch in train_loader:
        past_scene = batch["past_traj"].to(device)
        future_target = batch["future_target"].to(device)
        vehicle_mask = batch["vehicle_mask"].to(device)
        semantic_lane = build_semantic_lane_batch(batch, avm).to(device)

        optimizer.zero_grad()
        pred = model(past_scene, vehicle_mask, semantic_lane)
        loss = criterion(pred, future_target)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            total_loss += loss.item()
            total_ade += ade(pred, future_target).item()
            total_fde += fde(pred, future_target).item()
            n_batches += 1

    train_mse = total_loss / max(n_batches, 1)
    train_ade = total_ade / max(n_batches, 1)
    train_fde = total_fde / max(n_batches, 1)

    # -------- validation --------
    model.eval()
    val_loss = 0.0
    val_ade = 0.0
    val_fde = 0.0
    val_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            past_scene = batch["past_traj"].to(device)
            future_target = batch["future_target"].to(device)
            vehicle_mask = batch["vehicle_mask"].to(device)
            semantic_lane = build_semantic_lane_batch(batch, avm).to(device)

            pred = model(past_scene, vehicle_mask, semantic_lane)
            loss = criterion(pred, future_target)

            val_loss += loss.item()
            val_ade += ade(pred, future_target).item()
            val_fde += fde(pred, future_target).item()
            val_batches += 1

    val_mse = val_loss / max(val_batches, 1)
    val_ade = val_ade / max(val_batches, 1)
    val_fde = val_fde / max(val_batches, 1)

    print(
        f"Epoch {epoch+1}/{EPOCHS}  "
        f"train MSE: {train_mse:.6f}  ADE: {train_ade:.4f}  FDE: {train_fde:.4f}  "
        f"val MSE: {val_mse:.6f}  ADE: {val_ade:.4f}  FDE: {val_fde:.4f}"
    )

    metrics_log_semantic.append({
        "epoch": epoch + 1,
        "train_ade": train_ade,
        "train_fde": train_fde,
        "val_ade": val_ade,
        "val_fde": val_fde,
    })

print("BiGRU+Graph+Semantic training finished.")

In [ ]:
# 9. Save semantic model checkpoint and metrics to Drive
import json
import os

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

sem_ckpt_path = os.path.join(CHECKPOINT_DIR, "bigru_graph_semantic_model.pth")
print("Saving semantic model to:", sem_ckpt_path)

torch.save(model.state_dict(), sem_ckpt_path)

sem_full_ckpt_path = os.path.join(CHECKPOINT_DIR, "bigru_graph_semantic_checkpoint.pth")

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, sem_full_ckpt_path)

metrics_path = os.path.join(CHECKPOINT_DIR, "bigru_graph_semantic_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_log_semantic, f, indent=2)

print("Saved:")
print(" - weights:", sem_ckpt_path)
print(" - full checkpoint:", sem_full_ckpt_path)
print(" - metrics:", metrics_path)

In [ ]:
# 10. Quick qualitative check: plot one trajectory
from visualization import plot_trajectory

model.eval()
val_batch = next(iter(val_loader))
with torch.no_grad():
    semantic_lane = build_semantic_lane_batch(val_batch, avm).to(device)
    pred_val = model(
        val_batch["past_traj"].to(device),
        val_batch["vehicle_mask"].to(device),
        semantic_lane,
    ).cpu().numpy()

past_scene = val_batch["past_traj"].numpy()
future_target = val_batch["future_target"].numpy()

b = 0
past_target = past_scene[b, 0]
gt_future = future_target[b]
pred_future = pred_val[b]

plot_trajectory(past_target, gt_future, pred_future, title="BiGRU + Graph + Semantic prediction")